https://github.com/MIT-LCP/mimic-omop/blob/master/extras/concept/chart_label_to_concept.csv
https://github.com/YerevaNN/mimic3-benchmarks/blob/master/mimic3benchmark/resources/itemid_to_variable_map.csv

In [1]:
from __future__ import absolute_import
from __future__ import print_function
import pandas as pd
import csv
import sys
import os
import numpy as np
import shutil
pd.set_option('display.max_columns', 500)

def dataframe_from_csv(path, header=0, index_col=False):
    return pd.read_csv(path, header=header, index_col=index_col)


mimic_path = "C:\\Users\\김한재\\Desktop\\ONEASH_local\\Delirium-Prediction\\mostafaalishahi\\Delirium_prediction_models\\Data\\mimic-iii-clinical-database-1.4"
path_csv = "C:\\Users\\김한재\\Desktop\\ONEASH_local\\Delirium-Prediction\\mostafaalishahi\\Delirium_prediction_models\\Data\\admission_units"
data_path = "C:\\Users\\김한재\\Desktop\\ONEASH_local\\Delirium-Prediction\\mostafaalishahi\\Delirium_prediction_models\\Data\\preprocessed"

# Patient

In [9]:
patient = dataframe_from_csv(os.path.join(mimic_path, 'PATIENTS.csv'),index_col=False)
patient.drop(columns=['ROW_ID', 'DOD', 'DOD_HOSP', 'DOD_SSN', 'EXPIRE_FLAG'], inplace=True)

In [10]:
patient.groupby(['SUBJECT_ID']).head(1).shape

(46520, 3)

# ICU-Stay

In [11]:
icu = dataframe_from_csv(os.path.join(mimic_path, 'ICUSTAYS.csv'),index_col=False)
icu.drop(columns=['ROW_ID', 'FIRST_CAREUNIT', 'LAST_CAREUNIT', 'FIRST_WARDID', 'LAST_WARDID', 'DBSOURCE'], inplace=True)

In [12]:
icu.shape

(61532, 6)

In [13]:
icu.groupby(['SUBJECT_ID']).head(1).shape

(46476, 6)

# Filter ICU Stays on Age

In [14]:
patient_icu = pd.merge(icu, patient, on='SUBJECT_ID')

In [15]:
patient_icu.groupby(['SUBJECT_ID']).head(1).shape

(46476, 8)

In [16]:
patient_icu.groupby(['HADM_ID']).head(1).shape

(57786, 8)

In [17]:
patient_icu.shape

(61532, 8)

In [18]:
patient_icu['DOBYear'] = pd.to_datetime(patient_icu['DOB'], format='%Y-%m-%d %H:%M:%S')
patient_icu['DOBYear'] = patient_icu.DOBYear.dt.year
patient_icu['INTIMEYear'] = pd.to_datetime(patient_icu['INTIME'], format='%Y-%m-%d %H:%M:%S')
patient_icu['INTIMEYear'] = patient_icu.INTIMEYear.dt.year
patient_icu['AGE'] = patient_icu['INTIMEYear'] - patient_icu['DOBYear']
patient_icu.drop(columns=['DOBYear', 'INTIMEYear', 'DOB'], inplace=True)

In [19]:
patient_icu_under_1 = patient_icu[patient_icu.AGE <= 1]

In [20]:
patient_icu_under_1.groupby(['SUBJECT_ID']).head(1).shape

(7870, 8)

In [21]:
patient_icu_under_1.shape

(8100, 8)

In [22]:
patient_icu_adults = patient_icu[(patient_icu.AGE >= 18) & (patient_icu.AGE <= 89)]

In [23]:
patient_icu_adults.groupby(['SUBJECT_ID']).head(1).shape

(36548, 8)

In [24]:
patient_icu_adults.shape

(50641, 8)

In [25]:
patient_icu = patient_icu[(patient_icu.AGE >= 18) & (patient_icu.AGE <= 89)]

# Admission

In [26]:
admission = dataframe_from_csv(os.path.join(mimic_path, 'ADMISSIONS.csv'),index_col=False)
admission.drop(columns=['ROW_ID','ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'ADMISSION_TYPE', 'ADMISSION_LOCATION',
                        'DISCHARGE_LOCATION', 'EDREGTIME', 'EDOUTTIME', 'HAS_CHARTEVENTS_DATA', 'HOSPITAL_EXPIRE_FLAG',
                        'INSURANCE', 'LANGUAGE', 'RELIGION', 'MARITAL_STATUS'], inplace=True)

In [27]:
admission.head()

,SUBJECT_ID,HADM_ID,ETHNICITY,DIAGNOSIS
0,22,165315,WHITE,BENZODIAZEPINE OVERDOSE
1,23,152223,WHITE,CORONARY ARTERY DISEASE\CORONARY ARTERY BYPASS...
2,23,124321,WHITE,BRAIN MASS
3,24,161859,WHITE,INTERIOR MYOCARDIAL INFARCTION
4,25,129635,WHITE,ACUTE CORONARY SYNDROME


In [28]:
admission.shape

(58976, 4)

# Full ICUStays Information

In [29]:
adm_pat_icu = pd.merge(patient_icu, admission, on='HADM_ID')

In [30]:
adm_pat_icu.drop(columns=['SUBJECT_ID_y'], inplace=True)
col = ['SUBJECT_ID_x', 'HADM_ID', 'ICUSTAY_ID', 'GENDER', 'AGE', 'LOS', 'INTIME', 'OUTTIME', 'DIAGNOSIS','ETHNICITY']
adm_pat_icu = adm_pat_icu[col]
adm_pat_icu.columns = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'GENDER', 'AGE', 'LOS', 'INTIME', 'OUTTIME', 
                       'DIAGNOSIS', 'ETHNICITY']

In [31]:
g_map = {'F': 1, 'M': 2}
def transform_gender(gender_series):
    global g_map
    return {'GENDER': gender_series.fillna('').apply(lambda s: g_map[s] if s in g_map else g_map[''])}

In [32]:
adm_pat_icu['GENDER'] = adm_pat_icu['GENDER'].map({'F': 1, 'M': 2})

In [33]:
def transform_dx_into_id(df):
    df.DIAGNOSIS.fillna('nodx', inplace=True)
    dx_type = df.DIAGNOSIS.unique()
    dict_dx_key = pd.factorize(dx_type)[1]
    dict_dx_val = pd.factorize(dx_type)[0]
    dictionary  = dict(zip(dict_dx_key, dict_dx_val))
    df['DIAGNOSIS'] = df['DIAGNOSIS'].map(dictionary)
    return df

In [34]:
adm_pat_icu = transform_dx_into_id(adm_pat_icu)

C:\Users\김한재\AppData\Local\Temp\ipykernel_28596\1819538359.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df.DIAGNOSIS.fillna('nodx', inplace=True)


In [35]:
def transform_ethn_into_id(df):
    df.ETHNICITY.fillna('nodx', inplace=True)
    dx_type = df.ETHNICITY.unique()
    dict_dx_key = pd.factorize(dx_type)[1]
    dict_dx_val = pd.factorize(dx_type)[0]
    dictionary  = dict(zip(dict_dx_key, dict_dx_val))
    df['ETHNICITY'] = df['ETHNICITY'].map(dictionary)
    return df

In [36]:
adm_pat_icu = transform_ethn_into_id(adm_pat_icu)

C:\Users\김한재\AppData\Local\Temp\ipykernel_28596\3906134396.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df.ETHNICITY.fillna('nodx', inplace=True)


In [37]:
adm_pat_icu.head()

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,GENDER,AGE,LOS,INTIME,OUTTIME,DIAGNOSIS,ETHNICITY
0,268,110404,280836,1,66,3.2490,2198-02-14 23:27:38,2198-02-18 05:26:11,0.0,0
1,269,106296,206613,2,40,3.2788,2170-11-05 11:05:29,2170-11-08 17:46:57,1.0,1
2,270,188028,220345,2,80,2.8939,2128-06-24 15:05:20,2128-06-27 12:32:29,2.0,2
3,271,173727,249196,1,46,2.0600,2120-08-07 23:12:42,2120-08-10 00:39:04,3.0,3
4,272,164716,210407,2,67,1.6202,2186-12-25 21:08:04,2186-12-27 12:01:13,4.0,1


In [38]:
adm_pat_icu.shape

(50641, 10)

# Find ICUStays with Delirium from Chart

In [32]:
# Define delirium item ids
delirium_itemids = [228300, 228301, 228302, 228303, 228332, 228334, 228335, 228336, 228337]
chart_path = os.path.join(mimic_path, 'CHARTEVENTS.csv')

# Step 1: 먼저 delirium 관련 itemid에 해당하는 icustay_id만 뽑아서 set에 저장
icustay_delirium_set = set()
usecols_items = ['ICUSTAY_ID', 'ITEMID']
for chunk in pd.read_csv(chart_path, usecols=usecols_items, chunksize=10**6, low_memory=False):
    mask = chunk['ITEMID'].isin(delirium_itemids)
    icustay_ids = chunk.loc[mask, 'ICUSTAY_ID'].dropna().astype(int).unique()
    icustay_delirium_set.update(icustay_ids)
del icustay_ids, mask, chunk

# Step 2: 최종적으로 icustay_delirium_set 에 해당하는 icustay_id만 남긴 chart만 뽑는다.
# 메모리 절약을 위해 필요한 컬럼만 선택
drop_columns = ['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'STORETIME', 'CGID', 'RESULTSTATUS', 'STOPPED',
                'WARNING', 'ERROR']
chart_chunks = []
with pd.read_csv(chart_path, chunksize=10**6, low_memory=False) as reader:
    for chunk in reader:
        if 'ICUSTAY_ID' not in chunk.columns:
            continue
        chunk = chunk[chunk['ICUSTAY_ID'].notnull()]
        chunk['ICUSTAY_ID'] = chunk['ICUSTAY_ID'].astype(int)
        chart_filtered = chunk[chunk['ICUSTAY_ID'].isin(icustay_delirium_set)]
        chart_filtered = chart_filtered.drop(columns=[col for col in drop_columns if col in chart_filtered.columns], errors='ignore')
        chart_chunks.append(chart_filtered)
        del chart_filtered, chunk
chart = pd.concat(chart_chunks, ignore_index=True)
del chart_chunks


In [39]:
delirium = chart[chart.ITEMID.isin(delirium_itemids)]
icustay_delirium = delirium.ICUSTAY_ID.unique()

In [40]:
delirium.VALUE.unique()

array(['Negative', 'No', 'No (Stop - Not delirious)', 'Positive', 'Yes',
       'UTA', 'Unable to Assess', 'Yes (Continue)',
       'Yes (3 or more errors, then Continue)', 'Unable to Assess (Stop)',
       'No (less than 3 errors - Stop - Not delirious)'], dtype=object)

# D_ITEM - LABEL for other Tables

In [92]:
d_item = dataframe_from_csv(os.path.join(mimic_path, 'D_ITEMS.csv'),index_col=False)
d_item.drop(columns=['ABBREVIATION', 'CATEGORY', 'UNITNAME', 'CONCEPTID', 'ROW_ID', 'DBSOURCE', 'LINKSTO',
                     'PARAM_TYPE'], inplace=True) 

In [93]:
d_item.head()

,ITEMID,LABEL
0,497,Patient controlled analgesia (PCA) [Inject]
1,498,PCA Lockout (Min)
2,499,PCA Medication
3,500,PCA Total Dose
4,501,PCV Exh Vt (Obser)


# Add Item_id to chart 

In [43]:
chart = pd.merge(chart, d_item, on='ITEMID')

In [44]:
chart.head()

,ICUSTAY_ID,ITEMID,CHARTTIME,VALUE,VALUENUM,VALUEUOM,LABEL
0,291697,223751,2167-07-26 16:58:00,160.0,160.0,mmHg,Non-Invasive Blood Pressure Alarm - High
1,291697,223752,2167-07-26 16:58:00,90.0,90.0,mmHg,Non-Invasive Blood Pressure Alarm - Low
2,291697,223769,2167-07-26 16:58:00,100.0,100.0,%,O2 Saturation Pulseoxymetry Alarm - High
3,291697,223770,2167-07-26 16:58:00,92.0,92.0,%,O2 Saturation Pulseoxymetry Alarm - Low
4,291697,224161,2167-07-26 16:58:00,35.0,35.0,insp/min,Resp Alarm - High


In [45]:
chart.shape

(39657694, 7)

In [146]:
chart.to_csv(os.path.join(data_path, 'chartevent.csv'),index=False)

# Load Chart with Delirium ICU_Stays

In [2]:
chart = pd.read_csv(os.path.join(path_csv,'chartevent.csv'))

C:\Users\김한재\AppData\Local\Temp\ipykernel_28596\688850410.py:1: DtypeWarning: Columns (0: VALUE, 1: VALUEUOM) have mixed types. Specify dtype option on import or set low_memory=False.
  chart = pd.read_csv(os.path.join(path_csv,'chartevent.csv'))


# ICU-Stay shared between admission and chart 

In [39]:
icu_adm = adm_pat_icu.ICUSTAY_ID.unique()
icu_chart = chart.ICUSTAY_ID.unique()

In [40]:
def intersection(lst1, lst2): 
    lst3 = list(set(lst1).intersection(set(lst2)))
    return lst3 

In [41]:
icustay_delirium = intersection(icu_adm, icu_chart)

In [42]:
chart = chart[chart['ICUSTAY_ID'].isin(icustay_delirium)]

In [43]:
chart.loc[chart['VALUE'] == 'No'       , 'VALUENUM'] = 0
chart.loc[chart['VALUE'] == 'Negative' , 'VALUENUM'] = 0
chart.loc[chart['VALUE'] == 'No (Stop - Not delirious)' , 'VALUENUM'] = 0
chart.loc[chart['VALUE'] == 'Yes'     , 'VALUENUM'] = 1
chart.loc[chart['VALUE'] == 'Positive', 'VALUENUM'] = 1
chart.loc[chart['VALUE'] == 'Yes (Continue)', 'VALUENUM'] = 1
chart.loc[chart['VALUE'] == 'UTA'             , 'VALUENUM'] = 2
chart.loc[chart['VALUE'] == 'Unable to Assess', 'VALUENUM'] = 2
chart.loc[chart['VALUE'] == 'Unable to Assess (Stop)', 'VALUENUM'] = 2
chart.loc[chart['VALUE'] == 'No (less than 3 errors - Stop - Not delirious)', 'VALUENUM'] = 3
chart.loc[chart['VALUE'] == 'Yes (3 or more errors, then Continue)', 'VALUENUM'] = 4

In [44]:
chart.drop(columns=['VALUE'], inplace=True)
chart = chart.loc[chart.VALUENUM.notnull()]

In [45]:
chart.CHARTTIME = pd.to_datetime(chart.CHARTTIME)
chart.VALUEUOM = chart.VALUEUOM.fillna('').astype(str)
col = ['ICUSTAY_ID', 'ITEMID', 'LABEL', 'VALUENUM', 'VALUEUOM', 'CHARTTIME']
chart = chart[col]
chart.columns = ['ICUSTAY_ID', 'ITEMID', 'LABEL', 'VALUE', 'VALUEUOM', 'CHARTTIME']
chart[['ICUSTAY_ID']] = chart[['ICUSTAY_ID']].astype(int)

In [46]:
def check(x):
    try:
        x = float(str(x).strip())
    except:
        x = np.nan
    return x
def check_itemvalue(df):
    df['VALUE'] = df['VALUE'].apply(lambda x: check(x))
    df['VALUE'] = df.VALUE.astype(float)
    return df

In [47]:
chart = check_itemvalue(chart)
chart = chart.loc[chart.VALUE.notnull()]

In [48]:
chart_features = [ 
       'Delirium assessment',
       'CAM-ICU Inattention',
       'CAM-ICU Altered LOC',
       'CAM-ICU MS Change',
       'CAM-ICU MS change',
       'CAM-ICU RASS LOC',
       'CAM-ICU Disorganized thinking',
       'Admission Weight (Kg)' , 'Admission Weight (lbs.)', #Weight
       'Height (cm)', 'Admit Ht', 'Height Inches', 'Length Calc Inches', 'Length in Inches', 
       'Length Calc (cm)', 'Length in cm', #Height
       'O2 saturation pulseoxymetry', 'Arterial O2 Saturation', 'SpO2', # O2 Saturation
       'Heart Rate', #Heart
       'Temperature Fahrenheit', 'Temperature F', 'Temperature F (calc)', #Temperature
       'Temperature C (calc)', 'Temperature C', 'Temperature Celsius', #Temperatur
       'BUN', 'BUN (6-20)', #BUN
       'Glucose','Glucose (serum)','Glucose finger stick','Glucose (whole blood)','Glucose (70-105)',
       'Fingerstick Glucose', #Glucose
       'Hemoglobin', # serum hemoglobin
       'Platelets', #Platelets  
       'Inspired O2 Fraction', 'FiO2', # FiO2
       'Richmond-RAS Scale', 'Goal Richmond-RAS Scale', # Sedation Scor
       'Non Invasive Blood Pressure systolic', 'NBP [Systolic]',
       'Non Invasive Blood Pressure diastolic', 'NBP [Diastolic]',
       'Non Invasive Blood Pressure mean', 'NBP Mean',
       'Arterial Blood Pressure mean', 'Arterial BP Mean', 'ART BP mean',
       'Arterial Blood Pressure systolic', 'ART BP Systolic', 'Arterial BP [Systolic]',
       'Arterial Blood Pressure diastolic', 'Arterial BP [Diastolic]', 'ART BP Diastolic',
       'Arterial O2 pressure', 'Arterial PaO2', 'PAO2', #PaO2
       'Arterial CO2 Pressure', 'Arterial PaCO2', 'pCO2', 'pCO2 (other)', 'PCO2', 'Arterial PaCO2', #PaCo2
       'Respiratory Rate', #Respiratory    
       'PH (Arterial)','Arterial pH', 'Art.pH', 'pH (Art)', #pH      
       'GCS Total',
       'GCS - Eye Opening', 'Eye Opening',
       'GCS - Motor Response', 'Motor Response',
       'GCS - Verbal Response', 'Verbal Response',
       'TSH NML(0.27-4.2)', 'TSH  (NML 0.27-4.2)', #TSH
       'Serum Osmolality',
       'Ammonia', 'ammonia', 'AMMONIA', 'AMMONIA/12-47 UMOL/L',
       'Cortisol', 'cortisol',
       'ETCO2', 'EtCO2', 'EtCO2 Clinical indication',
       'dexmedetomidine', 'Dexmedetomidine',
       'Morphine Sulfate',
       'propofol', 'Propofol (Intubation)',
       'Midazolam/Versed', 'Midazolam',
       'Lorazepam/Ativan',
       'Fentanyl', 'fentanyl mg/hr', 'Fentanyl:', 'fentanyl',
       'Cardiac Index', 'CI (PiCCO)', 'Cardiac Index (CI NICOM)', 'cardiac index o',  #CI
       'Intra Cranial Pressure', 'Intra Cranial Pressure #2' ]


In [49]:
chart = chart[chart['LABEL'].isin(chart_features)]

In [50]:
chart[chart['LABEL'] == 'Admission Weight (Kg)'].shape

(7005, 6)

In [51]:
# lbs to Kg
chart.loc[chart['LABEL'] == 'Admission Weight (lbs.)' , 'VALUE'] = chart['VALUE'] * 0.45

# Inch to CM
chart.loc[chart['ITEMID'] == 920  , 'VALUE'] = chart['VALUE'] * 2.54
chart.loc[chart['ITEMID'] == 1394 , 'VALUE'] = chart['VALUE'] * 2.54
chart.loc[chart['ITEMID'] == 4187 , 'VALUE'] = chart['VALUE'] * 2.54
chart.loc[chart['ITEMID'] == 3486 , 'VALUE'] = chart['VALUE'] * 2.54

In [52]:
# Unifying several variables into one

# Weight
chart.loc[chart['LABEL'] == 'Admission Weight (Kg)'  , 'LABEL'] = 'Weight'
chart.loc[chart['LABEL'] == 'Admission Weight (lbs.)', 'LABEL'] = 'Weight'
# Height
chart.loc[chart['LABEL'] == 'Height (cm)'       , 'LABEL'] = 'Height'
chart.loc[chart['LABEL'] == 'Admit Ht'          , 'LABEL'] = 'Height'
chart.loc[chart['LABEL'] == 'Height Inches'     , 'LABEL'] = 'Height'
chart.loc[chart['LABEL'] == 'Length Calc Inches', 'LABEL'] = 'Height'
chart.loc[chart['LABEL'] == 'Length in Inches'  , 'LABEL'] = 'Height'
chart.loc[chart['LABEL'] == 'Length Calc (cm)'  , 'LABEL'] = 'Height'
chart.loc[chart['LABEL'] == 'Length in cm'      , 'LABEL'] = 'Height' 
# O2 Saturation
chart.loc[chart['LABEL'] == 'Arterial O2 Saturation'     , 'LABEL'] = 'Oxygen Saturation'
chart.loc[chart['LABEL'] == 'O2 saturation pulseoxymetry', 'LABEL'] = 'Oxygen Saturation'
chart.loc[chart['LABEL'] == 'SpO2', 'LABEL'] = 'Oxygen Saturation'
#Temperature
chart.loc[chart['LABEL'] == 'Temperature Fahrenheit', 'LABEL'] = 'Temperature F'
chart.loc[chart['LABEL'] == 'Temperature F (calc)'  , 'LABEL'] = 'Temperature F'

chart.loc[chart['LABEL'] == 'Temperature C (calc)'  , 'LABEL'] = 'Temperature C'
chart.loc[chart['LABEL'] == 'Temperature Celsius'  , 'LABEL'] = 'Temperature C'
# BUN
chart.loc[chart['LABEL'] == 'BUN (6-20)', 'LABEL'] = 'BUN'
# Glucose
chart.loc[chart['LABEL'] == 'Glucose (serum)'      , 'LABEL'] = 'Glucose'
chart.loc[chart['LABEL'] == 'Glucose finger stick' , 'LABEL'] = 'Glucose'
chart.loc[chart['LABEL'] == 'Glucose (whole blood)', 'LABEL'] = 'Glucose'
chart.loc[chart['LABEL'] == 'Glucose (70-105)'     , 'LABEL'] = 'Glucose'
chart.loc[chart['LABEL'] == 'Fingerstick Glucose'  , 'LABEL'] = 'Glucose'
chart.loc[chart['LABEL'] == 'Non Invasive Blood Pressure systolic' , 'LABEL'] = 'NBP [Systolic]'
chart.loc[chart['LABEL'] == 'Non Invasive Blood Pressure diastolic', 'LABEL'] = 'NBP [Diastolic]'
chart.loc[chart['LABEL'] == 'Non Invasive Blood Pressure mean'     , 'LABEL'] = 'NBP Mean'
chart.loc[chart['LABEL'] == 'Arterial Blood Pressure mean', 'LABEL'] = 'ART BP mean'
chart.loc[chart['LABEL'] == 'Arterial BP Mean'            , 'LABEL'] = 'ART BP mean'
chart.loc[chart['LABEL'] == 'Arterial Blood Pressure systolic', 'LABEL'] = 'ART BP Systolic'
chart.loc[chart['LABEL'] == 'Arterial BP [Systolic]'          , 'LABEL'] = 'ART BP Systolic'
chart.loc[chart['LABEL'] == 'Arterial Blood Pressure diastolic', 'LABEL'] = 'ART BP Diastolic'
chart.loc[chart['LABEL'] == 'Arterial BP [Diastolic]'          , 'LABEL'] = 'ART BP Diastolic'
chart.loc[chart['LABEL'] == 'Resp Rate (Spont)'    , 'LABEL'] = 'Respiratory Rate (spontaneous)'
chart.loc[chart['LABEL'] == 'Respiratory Rate Set' , 'LABEL'] = 'Respiratory Rate (Set)'
chart.loc[chart['LABEL'] == 'Resp Rate (Total)'    , 'LABEL'] = 'Respiratory Rate (Total)'
chart.loc[chart['LABEL'] == 'PH (Arterial)', 'LABEL'] = 'pH'
chart.loc[chart['LABEL'] == 'Arterial pH'  , 'LABEL'] = 'pH'
chart.loc[chart['LABEL'] == 'Art.pH'       , 'LABEL'] = 'pH' 
chart.loc[chart['LABEL'] == 'pH (Art)'  , 'LABEL'] = 'pH'
chart.loc[chart['LABEL'] == 'Eye Opening'    , 'LABEL'] = 'GCS - Eye Opening'
chart.loc[chart['LABEL'] == 'Motor Response' , 'LABEL'] = 'GCS - Motor Response'
chart.loc[chart['LABEL'] == 'Verbal Response', 'LABEL'] = 'GCS - Verbal Response'
chart.loc[chart['LABEL'] == 'Pain Level'     , 'LABEL'] = 'Pain Level Response'
chart.loc[chart['LABEL'] == 'Calcium (8.4-10.2)' , 'LABEL'] = 'Calcium'
chart.loc[chart['LABEL'] == 'Magnesium (1.6-2.6)', 'LABEL'] = 'Magnesium'
chart.loc[chart['LABEL'] == 'Arterial Base Excess', 'LABEL'] = 'Base Excess'
chart.loc[chart['LABEL'] == 'Arterial CO2(Calc)', 'LABEL'] = 'Total CO2'
chart.loc[chart['LABEL'] == 'Carbon Dioxide',     'LABEL'] = 'Total CO2'  
chart.loc[chart['LABEL'] == 'Anion gap', 'LABEL'] = 'Anion Gap'
chart.loc[chart['LABEL'] == 'O2 Flow (additional cannula)', 'LABEL'] = 'O2 Flow'
chart.loc[chart['LABEL'] == 'Arterial PaCO2',        'LABEL'] = 'PaCO2'
chart.loc[chart['LABEL'] == 'Arterial CO2 Pressure', 'LABEL'] = 'PaCO2'
chart.loc[chart['LABEL'] == 'pCO2',                  'LABEL'] = 'PaCO2'
chart.loc[chart['LABEL'] == 'pCO2 (other)',          'LABEL'] = 'PaCO2'
chart.loc[chart['LABEL'] == 'PCO2',                  'LABEL'] = 'PaCO2'
chart.loc[chart['LABEL'] == 'Arterial PaCO2',        'LABEL'] = 'PaCO2'
chart.loc[chart['LABEL'] == 'Arterial PaO2',        'LABEL'] = 'PaO2' 
chart.loc[chart['LABEL'] == 'Arterial O2 pressure', 'LABEL'] = 'PaO2'
chart.loc[chart['LABEL'] == 'PAO2',                 'LABEL'] = 'PaO2'
chart.loc[chart['LABEL'] == 'Inspired O2 Fraction', 'LABEL'] = 'FiO2'
chart.loc[chart['LABEL'] == 'Richmond-RAS Scale'      , 'LABEL'] = 'Sedation Score'
chart.loc[chart['LABEL'] == 'Goal Richmond-RAS Scale' , 'LABEL'] = 'Sedation Score'
chart.loc[chart['LABEL'] == 'Cardiac Index'   , 'LABEL'] = 'CI'
chart.loc[chart['LABEL'] == 'CI (PiCCO)'      , 'LABEL'] = 'CI'
chart.loc[chart['LABEL'] == 'cardiac index o'   , 'LABEL'] = 'CI'
chart.loc[chart['LABEL'] == 'Cardiac Index (CI NICOM)'  , 'LABEL'] = 'CI'  
chart.loc[chart['LABEL'] == 'Intra Cranial Pressure'      , 'LABEL'] = 'ICP'
chart.loc[chart['LABEL'] == 'Intra Cranial Pressure #2'   , 'LABEL'] = 'ICP'
chart.loc[chart['LABEL'] == 'TSH NML(0.27-4.2)'      , 'LABEL'] = 'TSH' 
chart.loc[chart['LABEL'] == 'TSH  (NML 0.27-4.2)'    , 'LABEL'] = 'TSH'
chart.loc[chart['LABEL'] == 'Serum Osmolality'    , 'LABEL'] = 'Osmolality'
chart.loc[chart['LABEL'] == 'ammonia',               'LABEL'] = 'Ammonia' 
chart.loc[chart['LABEL'] == 'AMMONIA',               'LABEL'] = 'Ammonia'
chart.loc[chart['LABEL'] == 'AMMONIA/12-47 UMOL/L',  'LABEL'] = 'Ammonia'
chart.loc[chart['LABEL'] == 'cortisol',    'LABEL'] = 'Cortisol' 
chart.loc[chart['LABEL'] == 'EtCO2',                      'LABEL'] = 'ETCO2' 
chart.loc[chart['LABEL'] == 'EtCO2 Clinical indication',  'LABEL'] = 'ETCO2'
chart.loc[chart['LABEL'] == 'dexmedetomidine',  'LABEL'] = 'Dexmedetomidine'  
chart.loc[chart['LABEL'] == 'propofol',               'LABEL'] = 'Propofol' 
chart.loc[chart['LABEL'] == 'Propofol (Intubation)',  'LABEL'] = 'Propofol'
chart.loc[chart['LABEL'] == 'Midazolam/Versed',  'LABEL'] = 'Midazolam'  
chart.loc[chart['LABEL'] == 'Midazolam',         'LABEL'] = 'Midazolam'
chart.loc[chart['LABEL'] == 'Lorazepam/Ativan',   'LABEL'] = 'Lorazepam'
chart.loc[chart['LABEL'] == 'fentanyl mg/hr',  'LABEL'] = 'Fentanyl' 
chart.loc[chart['LABEL'] == 'fentanyl',        'LABEL'] = 'Fentanyl'
chart.loc[chart['LABEL'] == 'Fentanyl:',       'LABEL'] = 'Fentanyl'

# LAB Events

In [54]:
lab = dataframe_from_csv(os.path.join(mimic_path, 'LABEVENTS.csv'),index_col=False)
lab.drop(columns=['ROW_ID', 'VALUENUM', 'FLAG'], inplace=True)

In [55]:
d_lab = dataframe_from_csv(os.path.join(mimic_path, 'D_LABITEMS.csv'),index_col=False)
d_lab.drop(columns=['ROW_ID', 'FLUID', 'CATEGORY', 'LOINC_CODE'], inplace=True)

In [56]:
lab_dlab = pd.merge(lab, d_lab, on='ITEMID')
lab_dlab.VALUEUOM = lab_dlab.VALUEUOM.fillna('').astype(str)
lab_dlab = lab_dlab.loc[lab_dlab.VALUE.notnull()]

In [57]:
lab_dlab.head()

,SUBJECT_ID,HADM_ID,ITEMID,CHARTTIME,VALUE,VALUEUOM,LABEL
0,3,NaN,50820,2101-10-12 16:07:00,7.39,units,pH
1,3,NaN,50800,2101-10-12 18:17:00,ART,,SPECIMEN TYPE
2,3,NaN,50802,2101-10-12 18:17:00,-1,mEq/L,Base Excess
3,3,NaN,50804,2101-10-12 18:17:00,22,mEq/L,Calculated Total CO2
4,3,NaN,50808,2101-10-12 18:17:00,0.93,mmol/L,Free Calcium


In [58]:
lab_dlab.shape

(27852706, 7)

# Add icu-stay to Lab events

In [59]:
icu_lab = pd.merge(lab_dlab, adm_pat_icu, how='right', on=['SUBJECT_ID'])

In [60]:
icu_lab.INTIME    = pd.to_datetime(icu_lab.INTIME)
icu_lab.OUTTIME   = pd.to_datetime(icu_lab.OUTTIME)
icu_lab.CHARTTIME = pd.to_datetime(icu_lab.CHARTTIME)

In [61]:
icu_lab = icu_lab[(icu_lab['CHARTTIME'] > icu_lab['INTIME']) & (icu_lab['CHARTTIME'] < icu_lab['OUTTIME'])]

In [62]:
icu_lab.drop(columns=['SUBJECT_ID', 'HADM_ID_y', 'HADM_ID_x', 'GENDER', 'AGE', 'LOS', 'INTIME', 'OUTTIME', 
                      'DIAGNOSIS', 'ETHNICITY'], inplace=True)
icu_lab[['ITEMID']] = icu_lab[['ITEMID']].astype(int)
col = ['ICUSTAY_ID', 'ITEMID', 'LABEL', 'VALUE', 'VALUEUOM', 'CHARTTIME']
icu_lab = icu_lab[col]

In [63]:
icu_lab = check_itemvalue(icu_lab)
icu_lab = icu_lab.loc[icu_lab.VALUE.notnull()]

In [64]:
icu_lab.head()

,ICUSTAY_ID,ITEMID,LABEL,VALUE,VALUEUOM,CHARTTIME
0,280836,51093,"Osmolality, Urine",374.0,mOsm/kg,2198-02-16 10:48:00
5,280836,51476,Epithelial Cells,0.0,#/hpf,2198-02-16 10:48:00
7,280836,51479,Granular Casts,10.0,#/lpf,2198-02-16 10:48:00
8,280836,51482,Hyaline Casts,86.0,#/lpf,2198-02-16 10:48:00
12,280836,51491,pH,5.0,units,2198-02-16 10:48:00


In [65]:
icu_lab.shape

(11171987, 6)

In [ ]:
icu_lab['LABEL'].unique()

<StringArray>
[         'Osmolality, Urine',           'Epithelial Cells',
             'Granular Casts',              'Hyaline Casts',
                         'pH',                    'Protein',
                        'RBC',           'Specific Gravity',
                        'WBC',                'Base Excess',
 ...
  'Miscellaneous, Body Fluid',       'Bicarbonate, Pleural',
                       'CD56',                          'H',
                'Urine Color', 'Reducing Substances, Urine',
                     'CD19 %',        'CD19 Absolute Count',
                     'CD20 %',        'CD20 Absolute Count']
Length: 389, dtype: str

In [71]:
icu_lab.head()

,ICUSTAY_ID,ITEMID,LABEL,VALUE,VALUEUOM,CHARTTIME
0,280836,51093,"Osmolality, Urine",374.0,mOsm/kg,2198-02-16 10:48:00
5,280836,51476,Epithelial Cells,0.0,#/hpf,2198-02-16 10:48:00
7,280836,51479,Granular Casts,10.0,#/lpf,2198-02-16 10:48:00
8,280836,51482,Hyaline Casts,86.0,#/lpf,2198-02-16 10:48:00
12,280836,51491,pH,5.0,units,2198-02-16 10:48:00


# Filter Lab Event based on Delirium icustay  and Features

In [72]:
icu_lab = icu_lab[icu_lab['ICUSTAY_ID'].isin(icustay_delirium)]

In [73]:
icu_lab[icu_lab['LABEL'].str.contains('Chloride')].shape

(52945, 6)

In [74]:
icu_lab_features = [
#Oxygen Saturation
        'Oxygen Saturation', 
#WBC
       'White Blood Cells', 'WBC', 'WBC Count', 
#Sodium
       'Sodium', 'Sodium, Whole Blood', 
# blood urea nitrogen (BUN)
        'Urea Nitrogen',
#Glucose   
    'Glucose', 
#Bilirubin
    'Bilirubin, Direct',# 'Bilirubin, Indirect', 'Bilirubin, Total', 
#Hemoglobin   
    'Hemoglobin',   
#Platelet
    'Platelet Count',
# Potassium
    'Potassium', 'Potassium, Whole Blood', 
# chloride
    'Chloride', 'Chloride, Whole Blood',
# Bicarbonate
       'Bicarbonate',
# Creatinine
       'Creatinine', 
# ALT
        'Alanine Aminotransferase (ALT)',
# AST
       'Asparate Aminotransferase (AST)', 
# Alkaline
       'Alkaline Phosphatase',
    'Thyroid Stimulating Hormone', #TSH
       'Osmolality, Measured', #serum osmolality
       'Carboxyhemoglobin', #Carboxyhemoglobin
       'SaO2', #Oxyhemoglobin
       'Methemoglobin', #Methemoglobin
       'Ammonia', #Ammonia
       'Cortisol', #Cortisol
       'Lactate', #Lactate
       'pH', #pH
       'pCO2', #pCO2
       'pO2' #pO2
] 

In [75]:
icu_lab = icu_lab[icu_lab['LABEL'].isin(icu_lab_features)]

In [76]:
icu_lab[icu_lab['LABEL'] == 'Hemoglobin'].shape

(42457, 6)

In [77]:
icu_lab.shape

(670376, 6)

## Unifying variables with more than one nam

In [82]:
icu_lab.loc[icu_lab['LABEL'] == 'White Blood Cells'     , 'LABEL'] = 'WBC'
icu_lab.loc[icu_lab['LABEL'] == 'WBC Count'             , 'LABEL'] = 'WBC'
#Sodium
icu_lab.loc[icu_lab['LABEL'] == 'Sodium, Whole Blood'   , 'LABEL'] = 'Sodium'
# blood urea nitrogen (BUN)
icu_lab.loc[icu_lab['LABEL'] == 'Urea Nitrogen'   , 'LABEL'] = 'BUN'
#Bilirubin
icu_lab.loc[icu_lab['LABEL'] == 'Bilirubin, Direct'     , 'LABEL'] = 'direct bilirubin'
icu_lab.loc[icu_lab['LABEL'] == 'Hematocrit'      , 'LABEL'] = 'Hemoglobin'
#Platelet
icu_lab.loc[icu_lab['LABEL'] == 'Platelet Count'      , 'LABEL'] = 'Platelets'
# Potassium
icu_lab.loc[icu_lab['LABEL'] == 'Potassium, Whole Blood', 'LABEL'] = 'Potassium'
# chloride
icu_lab.loc[icu_lab['LABEL'] == 'Chloride, Whole Blood' , 'LABEL'] = 'Chloride'
# ALT
icu_lab.loc[icu_lab['LABEL'] == 'Alanine Aminotransferase (ALT)' , 'LABEL'] = 'ALT'
# AST
icu_lab.loc[icu_lab['LABEL'] == 'Asparate Aminotransferase (AST)', 'LABEL'] = 'AST'
# Alkaline
icu_lab.loc[icu_lab['LABEL'] == 'Alkaline Phosphatase'  , 'LABEL'] = 'Alkaline Phosphate'
icu_lab.loc[icu_lab['LABEL'] == 'Myelocytes'            , 'LABEL'] = 'Metamyelocytes'
icu_lab.loc[icu_lab['LABEL'] == 'Calcium, Total'        , 'LABEL'] = 'Calcium'
icu_lab.loc[icu_lab['LABEL'] == 'Platelet Count'        , 'LABEL'] = 'Platelets'
icu_lab.loc[icu_lab['LABEL'] == 'Red Blood Cells'       , 'LABEL'] = 'RBC'  
icu_lab.loc[icu_lab['LABEL'] == 'pCO2'                  , 'LABEL'] = 'PaCO2'
icu_lab.loc[icu_lab['LABEL'] == 'pO2'                   , 'LABEL'] = 'PaO2'
icu_lab.loc[icu_lab['LABEL'] == 'Calculated Total CO2'  , 'LABEL'] = 'Total CO2'
icu_lab.loc[icu_lab['LABEL'] == 'Basophils'             , 'LABEL'] = 'Differential-Basos'
icu_lab.loc[icu_lab['LABEL'] == 'Eosinophils'           , 'LABEL'] = 'Differential-Eos'
icu_lab.loc[icu_lab['LABEL'] == 'Lymphocytes'           , 'LABEL'] = 'Differential-Lymphs'
icu_lab.loc[icu_lab['LABEL'] == 'Monocytes'             , 'LABEL'] = 'Differential-Monos'
icu_lab.loc[icu_lab['LABEL'] == 'Osmolality, Measured'  , 'LABEL'] = 'Osmolality'
icu_lab.loc[icu_lab['LABEL'] == 'SaO2'                  , 'LABEL'] = 'Oxyhemoglobin'

icu_lab.loc[icu_lab['LABEL'] == 'Thyroid Stimulating Hormone'    , 'LABEL'] = 'TSH'

In [83]:
icu_lab.head()

,ICUSTAY_ID,ITEMID,LABEL,VALUE,VALUEUOM,CHARTTIME
21115,246725,50882,Bicarbonate,23.0,mEq/L,2107-09-14 00:46:00
21117,246725,50902,Chloride,98.0,mEq/L,2107-09-14 00:46:00
21118,246725,50912,Creatinine,0.5,mg/dL,2107-09-14 00:46:00
21119,246725,50931,Glucose,138.0,mg/dL,2107-09-14 00:46:00
21122,246725,50971,Potassium,4.5,mEq/L,2107-09-14 00:46:00


In [84]:
icu_lab.shape

(670376, 6)

In [86]:
icu_lab.LABEL.unique()

<StringArray>
[       'Bicarbonate',           'Chloride',         'Creatinine',
            'Glucose',          'Potassium',             'Sodium',
                'BUN',                'TSH',         'Hemoglobin',
          'Platelets',                'WBC',                'ALT',
 'Alkaline Phosphate',                'AST',            'Lactate',
              'PaCO2',                 'pH',               'PaO2',
  'Oxygen Saturation',   'direct bilirubin',         'Osmolality',
           'Cortisol',  'Carboxyhemoglobin',      'Methemoglobin',
            'Ammonia']
Length: 25, dtype: str

In [185]:
icu_lab.to_csv(os.path.join(data_path, 'icu_lab.csv'),index=False)

In [186]:
icu_lab.head()

,ICUSTAY_ID,ITEMID,LABEL,VALUE,VALUEUOM,CHARTTIME
270,246725,50882,Bicarbonate,23.0,mEq/L,2107-09-14 00:46:00
272,246725,50902,Chloride,98.0,mEq/L,2107-09-14 00:46:00
273,246725,50912,Creatinine,0.5,mg/dL,2107-09-14 00:46:00
274,246725,50931,Glucose,138.0,mg/dL,2107-09-14 00:46:00
277,246725,50971,Potassium,4.5,mEq/L,2107-09-14 00:46:00


# Input-MV

In [88]:
input_mv = dataframe_from_csv(os.path.join(mimic_path, 'INPUTEVENTS_MV.csv'),index_col=False)
input_mv.drop(columns=['ORIGINALRATE', 'ORIGINALAMOUNT', 'COMMENTS_DATE', 'COMMENTS_CANCELEDBY', 'COMMENTS_EDITEDBY',
                       'CGID', 'RATE', 'RATEUOM', 'ORDERCATEGORYNAME', 'SECONDARYORDERCATEGORYNAME', 'LINKORDERID',
                       'ORDERCOMPONENTTYPEDESCRIPTION', 'ORDERCATEGORYDESCRIPTION', 'CONTINUEINNEXTDEPT', 'ORDERID',
                       'CANCELREASON', 'STATUSDESCRIPTION', 'ISOPENBAG', 'STORETIME', 'ENDTIME', 'ROW_ID','HADM_ID',
                       'SUBJECT_ID'], inplace=True)

In [89]:
input_mv.head()

,ICUSTAY_ID,STARTTIME,ITEMID,AMOUNT,AMOUNTUOM,PATIENTWEIGHT,TOTALAMOUNT,TOTALAMOUNTUOM
0,223259.0,2133-02-05 06:29:00,225166,6.774532,mEq,83.2,100.0,ml
1,223259.0,2133-02-05 05:34:00,225944,28.132997,ml,83.2,100.0,ml
2,223259.0,2133-02-05 05:34:00,225166,2.813300,mEq,83.2,100.0,ml
3,223259.0,2133-02-03 12:00:00,225893,1.000000,dose,83.2,100.0,ml
4,223259.0,2133-02-03 12:00:00,220949,100.000000,ml,83.2,100.0,ml


In [90]:
input_mv = input_mv.loc[input_mv.ICUSTAY_ID.notnull()]
input_mv = input_mv[input_mv['ICUSTAY_ID'].isin(icustay_delirium)]
input_mv.TOTALAMOUNTUOM = input_mv.TOTALAMOUNTUOM.fillna('').astype(str)
input_mv[['ICUSTAY_ID']] = input_mv[['ICUSTAY_ID']].astype(int)
input_mv.STARTTIME = pd.to_datetime(input_mv.STARTTIME)

In [94]:
input_mv = pd.merge(input_mv, d_item, on='ITEMID')

In [95]:
input_mv = input_mv[['ICUSTAY_ID', 'ITEMID', 'LABEL', 'AMOUNT', 'AMOUNTUOM', 'STARTTIME', 'PATIENTWEIGHT']]
input_mv.columns = ['ICUSTAY_ID', 'ITEMID', 'LABEL', 'VALUE', 'VALUEUOM', 'CHARTTIME', 'PATIENTWEIGHT']

In [96]:
input_mv.head()

,ICUSTAY_ID,ITEMID,LABEL,VALUE,VALUEUOM,CHARTTIME,PATIENTWEIGHT
0,281232,225158,NaCl 0.9%,14.844311,ml,2128-01-31 10:00:00,67.8
1,281232,225799,Gastric Meds,60.000000,ml,2128-01-31 14:35:00,67.8
2,281232,225975,Heparin Sodium (Prophylaxis),1.000000,dose,2128-02-23 08:00:00,67.8
3,281232,222042,Nicardipine,9.084546,mg,2128-02-14 10:40:00,67.8
4,281232,225158,NaCl 0.9%,45.422720,ml,2128-02-14 10:40:00,67.8


In [97]:
input_mv.shape

(1173074, 7)

In [98]:
input_mv = check_itemvalue(input_mv)
input_mv = input_mv.loc[input_mv.VALUE.notnull()]

In [99]:
input_mv_features = [
         'Dexmedetomidine (Precedex)',
         'Morphine Sulfate', 
         'Propofol',
         'Midazolam (Versed)', 'Midazolam',
         'Fentanyl', 'Fentanyl (Conc)', 'Fentanyl (Concentrate)', 'Fentanyl (Push)',
         'Lorazepam (Ativan)']

In [100]:
input_mv = input_mv[input_mv['LABEL'].isin(input_mv_features)]

In [101]:
input_mv.loc[input_mv['LABEL'] == 'Fentanyl (Conc)'          , 'LABEL'] = 'Fentanyl'
input_mv.loc[input_mv['LABEL'] == 'Fentanyl (Concentrate)'   , 'LABEL'] = 'Fentanyl'
input_mv.loc[input_mv['LABEL'] == 'Fentanyl (Push)'          , 'LABEL'] = 'Fentanyl'
input_mv.loc[input_mv['LABEL'] == 'Morphine Sulfate'  , 'LABEL'] = 'Morphine'
input_mv.loc[input_mv['LABEL'] == 'Midazolam (Versed)', 'LABEL'] = 'Midazolam' 
input_mv.loc[input_mv['LABEL'] == 'Lorazepam (Ativan)', 'LABEL'] = 'Lorazepam' 
input_mv.loc[input_mv['LABEL'] == 'Dexmedetomidine (Precedex)' , 'LABEL'] = 'Dexmedetomidine'
input_mv.loc[input_mv['LABEL'] == 'Insulin - Regular' , 'LABEL'] = 'Insulin'
input_mv.loc[input_mv['LABEL'] == 'Insulin - Humalog' , 'LABEL'] = 'Insulin'
input_mv.loc[input_mv['LABEL'] == 'Calcium Gluconate (CRRT)'   , 'LABEL'] = 'Calcium Gluconate'

# Filter ICUStay Infromation based on Delirium icustay
# Add weight from input-mv to adm-icu-pat

In [198]:
adm_pat_icu = adm_pat_icu[adm_pat_icu['ICUSTAY_ID'].isin(icustay_delirium)]

In [199]:
weight = input_mv.groupby(['ICUSTAY_ID']).head(1)[['ICUSTAY_ID', 'PATIENTWEIGHT']]

In [200]:
adm_pat_icu = pd.merge(adm_pat_icu, weight, how='left', on='ICUSTAY_ID')

In [201]:
adm_pat_icu.head()

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,GENDER,AGE,LOS,INTIME,OUTTIME,DIAGNOSIS,ETHNICITY,PATIENTWEIGHT_x,PATIENTWEIGHT_y
0,291,126219,246725,1,73,0.8276,2107-09-13 22:43:01,2107-09-14 18:34:48,22.0,1,NaN,NaN
1,85,112077,291697,2,77,1.9909,2167-07-25 18:50:37,2167-07-27 18:37:35,12.0,1,NaN,NaN
2,107,182383,252542,2,69,1.0806,2121-11-30 19:24:56,2121-12-01 21:20:57,98.0,0,NaN,NaN
3,107,174162,264253,2,70,1.9313,2122-05-14 19:38:27,2122-05-16 17:59:33,99.0,0,84.0,84.0
4,154,102354,201272,2,54,2.0859,2127-12-23 22:02:59,2127-12-26 00:06:44,144.0,1,82.1,82.1


In [202]:
adm_pat_icu.shape

(7292, 12)

In [203]:
adm_pat_icu.ICUSTAY_ID.nunique()

7292

# Drop weight from Input_mv

In [204]:
input_mv.drop(columns=['PATIENTWEIGHT'], inplace=True)

In [205]:
input_mv.head()

,ICUSTAY_ID,ITEMID,LABEL,VALUE,VALUEUOM,CHARTTIME
8,281232,221744,Fentanyl,49.999999,mcg,2128-02-06 15:47:00
11,281232,222168,Propofol,258.700601,mg,2128-01-25 04:06:00
36,281232,222168,Propofol,344.550018,mg,2128-01-31 10:28:00
40,281232,222168,Propofol,134.328358,mg,2128-01-30 10:50:00
46,281232,222168,Propofol,252.717400,mg,2128-03-10 13:18:00


In [206]:
input_mv.shape

(156487, 6)

In [207]:
input_mv.LABEL.unique()

<StringArray>
['Fentanyl', 'Propofol', 'Dexmedetomidine', 'Midazolam', 'Morphine',
 'Lorazepam']
Length: 6, dtype: str

# Prescriptions¶

In [208]:
prescription = dataframe_from_csv(os.path.join(mimic_path, 'PRESCRIPTIONS.csv'),index_col=False)
prescription.drop(columns=['ROW_ID', 'ENDDATE', 'DRUG_TYPE', 'DRUG_NAME_POE', 'DRUG_NAME_GENERIC',
                           'FORMULARY_DRUG_CD', 'NDC', 'ROUTE', 'FORM_UNIT_DISP', 'FORM_VAL_DISP', 'PROD_STRENGTH'],
                           inplace=True)

C:\Users\김한재\AppData\Local\Temp\ipykernel_15760\3390058500.py:12: DtypeWarning: Columns (0: GSN) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, header=header, index_col=index_col)


In [209]:
prescription = prescription.loc[prescription.DOSE_VAL_RX.notnull()]

In [210]:
col = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'GSN', 'DRUG', 'DOSE_VAL_RX', 'DOSE_UNIT_RX', 'STARTDATE']
prescription = prescription[col]
prescription.columns = ['SUBJECT_ID', 'HADM_ID','ICUSTAY_ID', 'ITEMID', 'LABEL', 'VALUE', 'VALUEUOM', 'CHARTTIME']

In [213]:
prescription.loc[prescription['ICUSTAY_ID'].isnull(), 'ICUSTAY_ID'] = 123456
prescription[['ICUSTAY_ID']] = prescription[['ICUSTAY_ID']].astype(int)
prescription.CHARTTIME = pd.to_datetime(prescription.CHARTTIME)

In [215]:
icu_copy = icu.copy()
icu_copy.drop(columns=['LOS'], inplace=True)
icu_copy['INTIME'] = pd.to_datetime(icu_copy['INTIME'], format= "%Y-%m-%d")
icu_copy['INTIME'] = icu_copy.INTIME.dt.date
icu_copy['OUTTIME'] = pd.to_datetime(icu_copy['OUTTIME'], format= "%Y-%m-%d")
icu_copy['OUTTIME'] = icu_copy.OUTTIME.dt.date

ValueError: unconverted data remains when parsing with format "%Y-%m-%d": " 23:27:38". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [216]:
prescription = pd.merge(prescription, icu_copy, how='left', on=['SUBJECT_ID', 'HADM_ID'])

In [217]:
prescription.CHARTTIME = pd.to_datetime(prescription.CHARTTIME)
prescription.INTIME    = pd.to_datetime(prescription.INTIME)
prescription.OUTTIME   = pd.to_datetime(prescription.OUTTIME)
prescription = prescription[(prescription['CHARTTIME'] >= prescription['INTIME']) & (prescription['CHARTTIME'] <= prescription['OUTTIME'])]

In [218]:
prescription.loc[prescription['ICUSTAY_ID_x'] == 123456, 'ICUSTAY_ID_x'] = np.nan
prescription.loc[prescription['ICUSTAY_ID_x'].isnull(), 'ICUSTAY_ID_x'] = prescription.ICUSTAY_ID_y

In [219]:
prescription.drop(columns=['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID_y', 'INTIME', 'OUTTIME'], inplace=True)
prescription.columns = ['ICUSTAY_ID', 'ITEMID', 'LABEL', 'VALUE', 'VALUEUOM', 'CHARTTIME']
prescription[['ICUSTAY_ID']] = prescription[['ICUSTAY_ID']].astype(int)
prescription = prescription[prescription['ICUSTAY_ID'].isin(icustay_delirium)]

In [221]:
prescription_copy = prescription.copy()
prescription = prescription.groupby('LABEL').fillna(method='ffill')
prescription['LABEL'] = prescription_copy.LABEL

AttributeError: 'DataFrameGroupBy' object has no attribute 'fillna'

In [222]:
prescription_copy = prescription.copy()
prescription = prescription.groupby('LABEL').fillna(method='bfill')
prescription['LABEL'] = prescription_copy.LABEL

AttributeError: 'DataFrameGroupBy' object has no attribute 'fillna'

In [223]:
col = ['ICUSTAY_ID', 'ITEMID', 'LABEL', 'VALUE', 'VALUEUOM', 'CHARTTIME']
prescription = prescription[col]

In [224]:
prescription = check_itemvalue(prescription)
prescription = prescription.loc[prescription.VALUE.notnull()]

In [225]:
prescription_features = [
         'Morphine Sulfate',
         'Propofol'
         'Lorazepam',
         'Midazolam'
]

In [226]:
prescription = prescription[prescription['LABEL'].isin(prescription_features)]

In [227]:
prescription.loc[prescription['LABEL'] == 'Heparin'   , 'LABEL'] = 'Heparin'
prescription.loc[prescription['LABEL'] == 'Lorazepam' , 'LABEL'] = 'Lorazepam'
prescription.loc[prescription['LABEL'] == 'Midazolam' , 'LABEL'] = 'Midazolam'
prescription.loc[prescription['LABEL'] == 'Propofol'  , 'LABEL'] = 'Propofol'
prescription.loc[prescription['LABEL'] == 'Morphine Sulfate'  , 'LABEL'] = 'Morphine'
prescription.loc[prescription['LABEL'] == 'Sodium Bicarbonate', 'LABEL'] = 'Sodium-Bicarbonate-prescrip'

In [228]:
prescription.head()

,ICUSTAY_ID,ITEMID,LABEL,VALUE,VALUEUOM,CHARTTIME
73280,239862,003779,Midazolam,2.0,mg,2137-10-27
73301,239862,051279,Midazolam,100.0,mg,2137-10-27
75643,287079,051279,Midazolam,100.0,mg,2137-10-20
227438,280720,051279,Midazolam,100.0,mg,2113-10-06
232727,280720,003779,Midazolam,2.0,mg,2113-10-08


In [229]:
prescription.shape

(2870, 6)

In [230]:
prescription.LABEL.unique()

<StringArray>
['Midazolam', 'Morphine']
Length: 2, dtype: str

# All Tables

In [231]:
tables = [chart, icu_lab, input_mv, prescription]
all_tables = pd.concat(tables)
all_tables[['ICUSTAY_ID']] = all_tables[['ICUSTAY_ID']].astype(int)
all_tables = all_tables.sort_values(by=['ICUSTAY_ID','CHARTTIME'], axis=0)
all_tables.reset_index(inplace=True, drop=True)

In [232]:
all_tables = check_itemvalue(all_tables)
all_tables = all_tables.loc[all_tables.VALUE.notnull()]

In [233]:
all_tables.head()

,ICUSTAY_ID,ITEMID,LABEL,VALUE,VALUEUOM,CHARTTIME
0,200001,50813,Lactate,1.40,mmol/L,2181-11-25 19:27:00
1,200001,50818,PaCO2,43.00,mm Hg,2181-11-25 19:27:00
2,200001,50820,pH,7.41,units,2181-11-25 19:27:00
3,200001,50821,PaO2,79.00,mm Hg,2181-11-25 19:27:00
4,200001,50822,Potassium,4.60,mEq/L,2181-11-25 19:27:00


In [234]:
all_tables.shape

(829733, 6)

In [137]:
chart.ICUSTAY_ID.nunique()

7292

In [138]:
all_tables.to_csv(os.path.join(data_path, 'all_tables.csv'),index=False)


# SAVE DATA

In [139]:
def cohort_stay_id(frame):
    cohort = frame.ICUSTAY_ID.unique()
    return cohort

# ADMISSION

In [142]:
def break_up_admission_by_unit_stay(adm, data_path, stayid, verbose=1):
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    for i, stay_id in enumerate(unit_stays):
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
        dn = os.path.join(data_path, str(stay_id))
        try:
            os.makedirs(dn)  
        except:
            pass
        adm.loc[adm.ICUSTAY_ID == stay_id].to_csv(os.path.join(dn, 'admission.csv'), index=False)
    if verbose:
        sys.stdout.write('DONE!\n')

In [143]:
stay_id_Adm  = cohort_stay_id(adm_pat_icu)

In [144]:
break_up_admission_by_unit_stay(adm_pat_icu, data_path, stayid=stay_id_Adm, verbose=1)

StayID 4366 of 7292...

KeyboardInterrupt: 

# CHART

In [ ]:
def break_up_chart_by_unit_stay(chart, data_path, stayid, verbose=1):
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    for i, stay_id in enumerate(unit_stays):
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
        dn = os.path.join(data_path, str(stay_id))
        try:
            os.makedirs(dn)  
        except:
            pass
        chart.loc[chart.ICUSTAY_ID == stay_id].to_csv(os.path.join(dn, 'chart.csv'), index=False)
    if verbose:
        sys.stdout.write('DONE!\n')

In [136]:
stay_id_chart  = cohort_stay_id(chart)

In [ ]:
break_up_chart_by_unit_stay(chart, data_path, stayid=stay_id_chart, verbose=1)

# ICU - LAB

In [ ]:
def break_up_icu_lab_by_unit_stay(icu_lab, data_path, stayid, verbose=1):
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    for i, stay_id in enumerate(unit_stays):
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
        dn = os.path.join(data_path, str(stay_id))
        try:
            os.makedirs(dn)  
        except:
            pass
        icu_lab.loc[icu_lab.ICUSTAY_ID == stay_id].to_csv(os.path.join(dn, 'icu_lab.csv'), index=False)
    if verbose:
        sys.stdout.write('DONE!\n')

In [ ]:
stay_id_icu_lab  = cohort_stay_id(icu_lab)

In [ ]:
break_up_icu_lab_by_unit_stay(icu_lab, data_path, stayid=stay_id_icu_lab, verbose=1)

# INPUT - MV

In [ ]:
def break_up_input_mv_by_unit_stay(input_mv, data_path, stayid, verbose=1):
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    for i, stay_id in enumerate(unit_stays):
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
        dn = os.path.join(data_path, str(stay_id))
        try:
            os.makedirs(dn)  
        except:
            pass
        input_mv.loc[input_mv.ICUSTAY_ID == stay_id].to_csv(os.path.join(dn, 'input_mv.csv'), index=False)
    if verbose:
        sys.stdout.write('DONE!\n')

In [ ]:
stay_id_input_mv  = cohort_stay_id(input_mv)

In [ ]:
break_up_input_mv_by_unit_stay(input_mv, data_path, stayid=stay_id_input_mv, verbose=1)

# PRESCRIPTION

In [ ]:
def break_up_prescription_by_unit_stay(prescription, data_path, stayid, verbose=1):
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    for i, stay_id in enumerate(unit_stays):
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
        dn = os.path.join(path_csv, str(stay_id))
        try:
            os.makedirs(dn)  
        except:
            pass
        prescription.loc[prescription.ICUSTAY_ID == stay_id].to_csv(os.path.join(dn, 'prescription.csv'), index=False)
    if verbose:
        sys.stdout.write('DONE!\n')

In [ ]:
stay_id_prescription  = cohort_stay_id(prescription)

In [ ]:
break_up_prescription_by_unit_stay(prescription, data_path, stayid=stay_id_prescription, verbose=1)

# ALL TABLES

In [ ]:
def break_up_all_tables_by_unit_stay(all_tables, data_path, stayid, verbose=1):
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    for i, stay_id in enumerate(unit_stays):
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
        dn = os.path.join(data_path, str(stay_id))
        try:
            os.makedirs(dn)  
        except:
            pass
        all_tables.loc[all_tables.ICUSTAY_ID == stay_id].to_csv(os.path.join(dn, 'all_tables.csv'), index=False)
    if verbose:
        sys.stdout.write('DONE!\n')

In [ ]:
stay_id_all_tables  = cohort_stay_id(all_tables)

In [ ]:
break_up_all_tables_by_unit_stay(all_tables, data_path, stayid=stay_id_all_tables, verbose=1)

# Extract Time Series

In [ ]:
all_variables = list(all_tables.LABEL.unique())

In [ ]:
all_variables

In [ ]:
def filter_on_variabels(all_tables, all_variables):
    all_tables = all_tables[all_tables['LABEL'].isin(all_variables)]
    return all_tables

In [ ]:
def convert_events_to_timeseries(all_features, all_variables):
    metadata  = all_features[['CHARTTIME', 'ICUSTAY_ID']].sort_values(by=['CHARTTIME'])\
                    .drop_duplicates(keep='first').set_index('CHARTTIME')
    timeserie = all_features[['CHARTTIME', 'LABEL', 'VALUE']]\
                    .sort_values(by=['CHARTTIME'], axis=0)\
                    .drop_duplicates(subset=['CHARTTIME', 'LABEL'], keep='last')
    time_piv  = timeserie.pivot(index='CHARTTIME', columns='LABEL', values='VALUE')
    timeseries = time_piv.merge(metadata, left_index=True, right_index=True).sort_index(axis=0).reset_index()
    for v in all_variables:
        if v not in timeseries.columns:
            timeseries[v] = np.nan            
    return timeseries

In [ ]:
def binning(final, x=60):
    final.CHARTTIME = pd.to_datetime(final.CHARTTIME)
    final.INTIME = pd.to_datetime(final.INTIME)
    final['HOURS'] = (final.CHARTTIME - final.INTIME).apply(lambda s: s / np.timedelta64(1, 's')) / 60./60
    final.drop(columns=['SUBJECT_ID', 'HADM_ID', 'CHARTTIME', 'INTIME', 'OUTTIME'], inplace=True)
    final['MINUTES'] = (final.HOURS).apply(lambda s: s * 60)
    final['BIN'] = (final['MINUTES']/ x).astype(int)
    final = final.fillna(final.groupby(['BIN']).transform('mean'))
    final.drop_duplicates(subset=['BIN'], keep='last',inplace=True)
    return final

In [ ]:
len(os.listdir(data_path))

In [ ]:
def extract_time_series_from_subject(data_path, all_variables):    
    for stay_dir in os.listdir(data_path):
        dn = os.path.join(data_path, stay_dir)
        try:
            sys.stdout.flush()
            
            admission  = dataframe_from_csv(os.path.join(data_path, stay_dir,'admission.csv'))
            all_tables = dataframe_from_csv(os.path.join(data_path, stay_dir, 'all_tables.csv'))
            all_tables = filter_on_variabels(all_tables, all_variables)
            all_features = all_tables.sort_values(by=['CHARTTIME'])
            timeepisode  = convert_events_to_timeseries(all_features, all_variables)
            final  = pd.merge(timeepisode, admission, on='ICUSTAY_ID')
            final  = final.sort_values(by=['CHARTTIME'])
            df_bin = binning(final, 60)
            df_bin.to_csv(os.path.join(dn, 'timeseries.csv'), index=False)
            sys.stdout.write('\rWrite StayID {0}...\n'.format(int(stay_dir)))
            if not os.path.isfile(os.path.join(dn,'timeseries.csv')):
                raise Exception
        except :
            continue
    print('DONE')

In [ ]:
extract_time_series_from_subject(data_path, all_variables)

In [161]:
import shutil
def delete_wo_timeseries(t_path):
    # import pdb;pdb.set_trace()
    for stay_dir in os.listdir(t_path):
        dn = os.path.join(t_path, stay_dir)
        try:
            stay_id = int(stay_dir)
            if not os.path.isdir(dn):
                raise Exception
        except:
            continue
        try:
            sys.stdout.flush()
            if not os.path.isfile(os.path.join(dn, 'timeseries.csv')):
                shutil.rmtree(dn)
        except:
            continue
    print('DONE deleting')

In [ ]:
delete_wo_timeseries(data_path)

In [ ]:
len(os.listdir(data_path))

In [ ]:
all_stays  = pd.Series(os.listdir(data_path))
all_filenames = []
for stay_id in (all_stays):
    df_file = os.path.join(data_path, str(stay_id), 'timeseries.csv')
    all_filenames.append(df_file)
all_series = pd.concat([pd.read_csv(f) for f in all_filenames])

In [ ]:
combined_csv = pd.concat([pd.read_csv(f) for f in all_filenames])

combined_csv.to_csv(os.path.join(path_csv, 'all_data_delirium_mimic.csv'), index=False)
